<a href="https://colab.research.google.com/github/Alisafi66/ABDELKERIM-ALI-HASSAN-/blob/main/Algorithmic_Fairness_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Algorithmic Fairness 2




## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os

## 2. Load Data

In [2]:
df = pd.read_csv('adult_train.csv', na_values=['?', ' ?'])
print("Shape:", df.shape)
display(df.head())

Shape: (32561, 15)


,Age,Workclass,fnlwgt,Education,Education_Num,Martial_Status,Occupation,Relationship,Race,Sex,Capital_Gain,Capital_Loss,Hours_per_week,Country,Target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


# Dataset Preprocessing
## Load & Clean

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('adult_train.csv')
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
df = df.replace('?', pd.NA)
df = df.dropna().reset_index(drop=True)

## Drop Irrelevant Columns
Dropping: `fnlwgt`, `Education_Num`, `Capital_Gain`, `Capital_Loss`

In [4]:
cols_to_drop = ['fnlwgt', 'Education_Num', 'Capital_Gain', 'Capital_Loss']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

## Education
Mapping grade levels to readable categories: Elementary, Middle_School, High_School, High_School_Grad, Some_College, Associate, Bachelors, Masters, Professional, Doctorate

In [5]:
education_map = {
    'Preschool':'Preschool','1st-4th':'Elementary','5th-6th':'Elementary',
    '7th-8th':'Middle_School','9th':'Middle_School','10th':'High_School',
    '11th':'High_School','12th':'High_School','HS-grad':'High_School_Grad',
    'Some-college':'Some_College','Assoc-voc':'Associate','Assoc-acdm':'Associate',
    'Bachelors':'Bachelors','Masters':'Masters','Prof-school':'Professional',
    'Doctorate':'Doctorate'
}
df['Education'] = df['Education'].map(education_map)

## Martial_Status
Grouping: Married-civ-spouse + Married-AF-spouse → Married | Divorced + Separated + Married-spouse-absent → Separated | Never-married → Single

In [6]:
marital_map = {
    'Married-civ-spouse':'Married','Married-AF-spouse':'Married',
    'Married-spouse-absent':'Separated','Divorced':'Separated',
    'Separated':'Separated','Widowed':'Widowed','Never-married':'Single'
}
df['Martial_Status'] = df['Martial_Status'].map(marital_map)

## Relationship
Grouping: Husband + Wife → Married | Not-in-family + Unmarried + Own-child → Single | Other-relative → Other

In [7]:
relationship_map = {
    'Husband':'Married','Wife':'Married',
    'Not-in-family':'Single','Unmarried':'Single',
    'Own-child':'Single','Other-relative':'Other'
}
df['Relationship'] = df['Relationship'].map(relationship_map)

## Hours_per_week
Grouping: <= 20 → Part_Time | 21–30 → Part_Time | 31–40 → Full_Time | > 40 → Over_Time

In [8]:
def categorize_hours(h):
    if h <= 20:   return 'Part_Time'
    elif h <= 30: return 'Part_Time'
    elif h <= 40: return 'Full_Time'
    else:         return 'Over_Time'

df['Hours_per_week'] = df['Hours_per_week'].apply(categorize_hours)

## Workclass
Grouping: Self-emp-not-inc + Self-emp-inc → Self_Employed | Local-gov + State-gov + Federal-gov → Government

In [9]:
workclass_map = {
    'Private':'Private','Self-emp-not-inc':'Self_Employed',
    'Self-emp-inc':'Self_Employed','Local-gov':'Government',
    'State-gov':'Government','Federal-gov':'Government',
    'Without-pay':'Without_Pay'
}
df['Workclass'] = df['Workclass'].map(workclass_map)

## Country
Grouping: United-States → United_States | All others → Other

In [10]:
df['Country'] = df['Country'].apply(lambda x: 'United_States' if x == 'United-States' else 'Other')

## Occupation
Renaming hyphenated values to clean readable labels

In [11]:
occupation_map = {
    'Prof-specialty':'Professional','Craft-repair':'Craft_Repair',
    'Exec-managerial':'Executive','Adm-clerical':'Administrative',
    'Sales':'Sales','Other-service':'Other_Service',
    'Machine-op-inspct':'Machine_Operator','Transport-moving':'Transport',
    'Handlers-cleaners':'Handlers','Farming-fishing':'Farming',
    'Tech-support':'Tech_Support','Protective-serv':'Protective_Service',
    'Priv-house-serv':'House_Service','Armed-Forces':'Armed_Forces'
}
df['Occupation'] = df['Occupation'].map(occupation_map)

## Final Verification

In [12]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nNaN check:\n", df.isna().sum())
display(df.head(50))

Shape: (30162, 11)

Columns: ['Age', 'Workclass', 'Education', 'Martial_Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Hours_per_week', 'Country', 'Target']

NaN check:
 Age               0
Workclass         0
Education         0
Martial_Status    0
Occupation        0
Relationship      0
Race              0
Sex               0
Hours_per_week    0
Country           0
Target            0
dtype: int64


,Age,Workclass,Education,Martial_Status,Occupation,Relationship,Race,Sex,Hours_per_week,Country,Target
0,39,Government,Bachelors,Single,Administrative,Single,White,Male,Full_Time,United_States,<=50K
1,50,Self_Employed,Bachelors,Married,Executive,Married,White,Male,Part_Time,United_States,<=50K
2,38,Private,High_School_Grad,Separated,Handlers,Single,White,Male,Full_Time,United_States,<=50K
3,53,Private,High_School,Married,Handlers,Married,Black,Male,Full_Time,United_States,<=50K
4,28,Private,Bachelors,Married,Professional,Married,Black,Female,Full_Time,Other,<=50K
5,37,Private,Masters,Married,Executive,Married,White,Female,Full_Time,United_States,<=50K
6,49,Private,Middle_School,Separated,Other_Service,Single,Black,Female,Part_Time,Other,<=50K
7,52,Self_Employed,High_School_Grad,Married,Executive,Married,White,Male,Over_Time,United_States,>50K
8,31,Private,Masters,Single,Professional,Single,White,Female,Over_Time,United_States,>50K
9,42,Private,Bachelors,Married,Executive,Married,White,Male,Full_Time,United_States,>50K


# Encoding & Split

In [16]:
# Save sensitive attributes before encoding
sensitive_attrs = df[['Sex', 'Race']].copy()

# Binarize target — handle both string and int
if df['Target'].dtype == object:
    df['Target'] = df['Target'].apply(lambda x: 1 if '>50K' in x else 0)
else:
    df['Target'] = df['Target'].apply(lambda x: 1 if x == 1 else 0)

# Encode all categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# Split
X = df.drop(columns=['Target'])
y = df['Target']

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive_attrs,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale
scaler_with = StandardScaler()
X_train_with = scaler_with.fit_transform(X_train)
X_test_with  = scaler_with.transform(X_test)

X_train_no_sens = X_train.drop(columns=['Sex', 'Race'])
X_test_no_sens  = X_test.drop(columns=['Sex', 'Race'])

scaler_without = StandardScaler()
X_train_without = scaler_without.fit_transform(X_train_no_sens)
X_test_without  = scaler_without.transform(X_test_no_sens)

print("Train size:", X_train.shape[0])
print("Test size: ", X_test.shape[0])
print("Features with sensitive attributes:   ", X_train.shape[1])
print("Features without sensitive attributes:", X_train_no_sens.shape[1])

Train size: 24129
Test size:  6033
Features with sensitive attributes:    10
Features without sensitive attributes: 8


# Model Training

In [17]:
lr_with    = LogisticRegression(max_iter=1000, random_state=42)
lr_without = LogisticRegression(max_iter=1000, random_state=42)
lr_with.fit(X_train_with, y_train)
lr_without.fit(X_train_without, y_train)

rf_with    = RandomForestClassifier(n_estimators=100, random_state=42)
rf_without = RandomForestClassifier(n_estimators=100, random_state=42)
rf_with.fit(X_train_with, y_train)
rf_without.fit(X_train_without, y_train)

dt_with    = DecisionTreeClassifier(random_state=42)
dt_without = DecisionTreeClassifier(random_state=42)
dt_with.fit(X_train_with, y_train)
dt_without.fit(X_train_without, y_train)

y_pred_lr = lr_with.predict(X_test_with)
y_pred_rf = rf_with.predict(X_test_with)
y_pred_dt = dt_with.predict(X_test_with)

print("All models trained.")
print("LR  predictions:", y_pred_lr.shape)
print("RF  predictions:", y_pred_rf.shape)
print("DT  predictions:", y_pred_dt.shape)

All models trained.
LR  predictions: (6033,)
RF  predictions: (6033,)
DT  predictions: (6033,)


# Disparate Treatment — Logistic Regression

In [18]:
from itertools import combinations

non_sensitive_cols = ['Age', 'Workclass', 'Education', 'Martial_Status',
                      'Occupation', 'Relationship', 'Hours_per_week', 'Country']

test_df = X_test.copy().reset_index(drop=True)
test_df['Sex']        = sens_test.reset_index(drop=True)['Sex']
test_df['Race']       = sens_test.reset_index(drop=True)['Race']
test_df['prediction'] = lr_with.predict(X_test_with)
print("  Logistic Regression — Disparate Treatment")

# Mappings for encoded sensitive attributes (deduced from kernel state and previous outputs)
sex_encoded_map = {'Male': 1, 'Female': 0}
race_display_map = {0: 'Amer-Indian-Eskimo', 1: 'Asian-Pac-Islander', 2: 'Black', 3: 'Other', 4: 'White'}

# SEX
sex_groups = test_df.groupby(non_sensitive_cols)
total_pairs   = 0
diff_pred_pairs = 0

for _, group in sex_groups:
    # Use encoded integer values for filtering 'Sex'
    males   = group[group['Sex'] == sex_encoded_map['Male']]
    females = group[group['Sex'] == sex_encoded_map['Female']]
    if len(males) > 0 and len(females) > 0:
        for _, m in males.iterrows():
            for _, f in females.iterrows():
                total_pairs += 1
                if m['prediction'] != f['prediction']:
                    diff_pred_pairs += 1

pct = (diff_pred_pairs / total_pairs * 100) if total_pairs > 0 else 0
verdict = "DISPARATE TREATMENT DETECTED" if diff_pred_pairs > 0 else "NO Disparate Treatment"

sex_summary = pd.DataFrame([{
    'Group 1'    : 'Male',
    'Group 2'    : 'Female',
    'Total Pairs': total_pairs,
    'Diff Preds' : diff_pred_pairs,
    '% Affected' : round(pct, 2),
    'Verdict'    : verdict
}])
print("\nSex Summary Table:")
display(sex_summary)

print(f"  Total Male/Female pairs with same non-sensitive attributes : {total_pairs}")
print(f"  Pairs with different predictions                           : {diff_pred_pairs}")
print(f"  % affected                                                 : {pct:.2f}%")
print(f"  Verdict                                                    : {verdict}")

# RACE
print("\n RACE (All group pairs) ")
race_groups_list = test_df['Race'].unique()
race_pairs = list(combinations(race_groups_list, 2))

race_summary = []
for r1, r2 in race_pairs:
    total_pairs_race = 0
    diff_pred_race   = 0
    for _, group in sex_groups:
        g1 = group[group['Race'] == r1]
        g2 = group[group['Race'] == r2]
        if len(g1) > 0 and len(g2) > 0:
            for _, row1 in g1.iterrows():
                for _, row2 in g2.iterrows():
                    total_pairs_race += 1
                    if row1['prediction'] != row2['prediction']:
                        diff_pred_race += 1
    pct_race = (diff_pred_race / total_pairs_race * 100) if total_pairs_race > 0 else 0
    verdict_race = "DETECTED" if diff_pred_race > 0 else "NO Disparate Treatment"
    race_summary.append({
        'Group 1'       : r1,
        'Group 2'       : r2,
        'Total Pairs'   : total_pairs_race,
        'Diff Preds'    : diff_pred_race,
        '% Affected'    : round(pct_race, 2),
        'Verdict'       : verdict_race
    })
    # Convert integer race codes to string names for printing
    print(f"  {race_display_map[r1]:25s} vs {race_display_map[r2]:25s} | Pairs: {total_pairs_race:4d} | Diff: {diff_pred_race:4d} | {pct_race:.2f}% | {verdict_race}")

print("\nRace Summary Table:")
display(pd.DataFrame(race_summary))

  Logistic Regression — Disparate Treatment

Sex Summary Table:


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Male,Female,799,16,2.0,DISPARATE TREATMENT DETECTED


  Total Male/Female pairs with same non-sensitive attributes : 799
  Pairs with different predictions                           : 16
  % affected                                                 : 2.00%
  Verdict                                                    : DISPARATE TREATMENT DETECTED

 RACE (All group pairs) 
  White                     vs Other                     | Pairs:   23 | Diff:    0 | 0.00% | NO Disparate Treatment
  White                     vs Black                     | Pairs:  340 | Diff:   13 | 3.82% | DETECTED
  White                     vs Amer-Indian-Eskimo        | Pairs:   39 | Diff:    1 | 2.56% | DETECTED
  White                     vs Asian-Pac-Islander        | Pairs:   70 | Diff:    4 | 5.71% | DETECTED
  Other                     vs Black                     | Pairs:    2 | Diff:    0 | 0.00% | NO Disparate Treatment
  Other                     vs Amer-Indian-Eskimo        | Pairs:    1 | Diff:    0 | 0.00% | NO Disparate Treatment
  Other             

,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,4,3,23,0,0.00,NO Disparate Treatment
1,4,2,340,13,3.82,DETECTED
2,4,0,39,1,2.56,DETECTED
3,4,1,70,4,5.71,DETECTED
4,3,2,2,0,0.00,NO Disparate Treatment
5,3,0,1,0,0.00,NO Disparate Treatment
6,3,1,2,0,0.00,NO Disparate Treatment
7,2,0,5,0,0.00,NO Disparate Treatment
8,2,1,2,0,0.00,NO Disparate Treatment
9,0,1,1,0,0.00,NO Disparate Treatment


# Disparate Treatment — Random Forest

In [20]:
from itertools import combinations

test_df['prediction'] = rf_with.predict(X_test_with)

print("="*60)
print("  Random Forest — Disparate Treatment")
print("="*60)

# Mappings for encoded sensitive attributes (deduced from kernel state and previous outputs)
sex_encoded_map = {'Male': 1, 'Female': 0}
race_display_map = {0: 'Amer-Indian-Eskimo', 1: 'Asian-Pac-Islander', 2: 'Black', 3: 'Other', 4: 'White'}

print("\n--- SEX (Male vs Female) ---")
sex_groups    = test_df.groupby(non_sensitive_cols)
total_pairs   = 0
diff_pred_pairs = 0

for _, group in sex_groups:
    # Use encoded integer values for filtering 'Sex'
    males   = group[group['Sex'] == sex_encoded_map['Male']]
    females = group[group['Sex'] == sex_encoded_map['Female']]
    if len(males) > 0 and len(females) > 0:
        for _, m in males.iterrows():
            for _, f in females.iterrows():
                total_pairs += 1
                if m['prediction'] != f['prediction']:
                    diff_pred_pairs += 1

pct     = (diff_pred_pairs / total_pairs * 100) if total_pairs > 0 else 0
verdict = "DISPARATE TREATMENT DETECTED" if diff_pred_pairs > 0 else "NO Disparate Treatment"

sex_summary = pd.DataFrame([{
    'Group 1'    : 'Male',
    'Group 2'    : 'Female',
    'Total Pairs': total_pairs,
    'Diff Preds' : diff_pred_pairs,
    '% Affected' : round(pct, 2),
    'Verdict'    : verdict
}])
print("\nSex Summary Table:")
display(sex_summary)

print(f"  Total Male/Female pairs with same non-sensitive attributes : {total_pairs}")
print(f"  Pairs with different predictions                           : {diff_pred_pairs}")
print(f"  % affected                                                 : {pct:.2f}%")
print(f"  Verdict                                                    : {verdict}")

print("\n--- RACE (All group pairs) ---")
race_groups_list = test_df['Race'].unique()
race_pairs       = list(combinations(race_groups_list, 2))

race_summary = []
for r1, r2 in race_pairs:
    total_pairs_race = 0
    diff_pred_race   = 0
    for _, group in sex_groups:
        g1 = group[group['Race'] == r1]
        g2 = group[group['Race'] == r2]
        if len(g1) > 0 and len(g2) > 0:
            for _, row1 in g1.iterrows():
                for _, row2 in g2.iterrows():
                    total_pairs_race += 1
                    if row1['prediction'] != row2['prediction']:
                        diff_pred_race += 1
    pct_race     = (diff_pred_race / total_pairs_race * 100) if total_pairs_race > 0 else 0
    verdict_race = "DETECTED" if diff_pred_race > 0 else "NO Disparate Treatment"
    race_summary.append({
        'Group 1'    : r1,
        'Group 2'    : r2,
        'Total Pairs': total_pairs_race,
        'Diff Preds' : diff_pred_race,
        '% Affected' : round(pct_race, 2),
        'Verdict'    : verdict_race
    })
    # Convert integer race codes to string names for printing
    print(f"  {race_display_map[r1]:25s} vs {race_display_map[r2]:25s} | Pairs: {total_pairs_race:4d} | Diff: {diff_pred_race:4d} | {pct_race:.2f}% | {verdict_race}")

print("\nRace Summary Table:")
display(pd.DataFrame(race_summary))

  Random Forest — Disparate Treatment

--- SEX (Male vs Female) ---

Sex Summary Table:


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Male,Female,799,31,3.88,DISPARATE TREATMENT DETECTED


  Total Male/Female pairs with same non-sensitive attributes : 799
  Pairs with different predictions                           : 31
  % affected                                                 : 3.88%
  Verdict                                                    : DISPARATE TREATMENT DETECTED

--- RACE (All group pairs) ---
  White                     vs Other                     | Pairs:   23 | Diff:    1 | 4.35% | DETECTED
  White                     vs Black                     | Pairs:  340 | Diff:   25 | 7.35% | DETECTED
  White                     vs Amer-Indian-Eskimo        | Pairs:   39 | Diff:    1 | 2.56% | DETECTED
  White                     vs Asian-Pac-Islander        | Pairs:   70 | Diff:    4 | 5.71% | DETECTED
  Other                     vs Black                     | Pairs:    2 | Diff:    0 | 0.00% | NO Disparate Treatment
  Other                     vs Amer-Indian-Eskimo        | Pairs:    1 | Diff:    0 | 0.00% | NO Disparate Treatment
  Other                     

,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,4,3,23,1,4.35,DETECTED
1,4,2,340,25,7.35,DETECTED
2,4,0,39,1,2.56,DETECTED
3,4,1,70,4,5.71,DETECTED
4,3,2,2,0,0.00,NO Disparate Treatment
5,3,0,1,0,0.00,NO Disparate Treatment
6,3,1,2,0,0.00,NO Disparate Treatment
7,2,0,5,0,0.00,NO Disparate Treatment
8,2,1,2,0,0.00,NO Disparate Treatment
9,0,1,1,0,0.00,NO Disparate Treatment


# Disparate Treatment — Decision Tree

In [21]:
test_df['prediction'] = dt_with.predict(X_test_with)
sex_groups = test_df.groupby(non_sensitive_cols)

print("="*60)
print("  Decision Tree — Disparate Treatment")
print("="*60)

print("\n--- SEX (Male vs Female) ---")
total_pairs     = 0
diff_pred_pairs = 0

for _, group in sex_groups:
    males   = group[group['Sex'] == sex_encoded_map['Male']]
    females = group[group['Sex'] == sex_encoded_map['Female']]
    if len(males) > 0 and len(females) > 0:
        for _, m in males.iterrows():
            for _, f in females.iterrows():
                total_pairs += 1
                if m['prediction'] != f['prediction']:
                    diff_pred_pairs += 1

pct     = (diff_pred_pairs / total_pairs * 100) if total_pairs > 0 else 0
verdict = "DETECTED" if diff_pred_pairs > 0 else "NO Disparate Treatment"

display(pd.DataFrame([{
    'Group 1'    : 'Male',
    'Group 2'    : 'Female',
    'Total Pairs': total_pairs,
    'Diff Preds' : diff_pred_pairs,
    '% Affected' : round(pct, 2),
    'Verdict'    : verdict
}]))

print("\n--- RACE (All group pairs) ---")
race_groups_list = sorted(test_df['Race'].unique())
race_pairs_list  = list(combinations(race_groups_list, 2))

race_summary = []
for r1, r2 in race_pairs_list:
    total_r = 0
    diff_r  = 0
    for _, group in sex_groups:
        g1 = group[group['Race'] == r1]
        g2 = group[group['Race'] == r2]
        if len(g1) > 0 and len(g2) > 0:
            for _, row1 in g1.iterrows():
                for _, row2 in g2.iterrows():
                    total_r += 1
                    if row1['prediction'] != row2['prediction']:
                        diff_r += 1
    pct_r     = (diff_r / total_r * 100) if total_r > 0 else 0
    verdict_r = "DETECTED" if diff_r > 0 else "NO Disparate Treatment"
    race_summary.append({
        'Group 1'    : race_display_map[r1],
        'Group 2'    : race_display_map[r2],
        'Total Pairs': total_r,
        'Diff Preds' : diff_r,
        '% Affected' : round(pct_r, 2),
        'Verdict'    : verdict_r
    })

display(pd.DataFrame(race_summary))

  Decision Tree — Disparate Treatment

--- SEX (Male vs Female) ---


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Male,Female,799,33,4.13,DETECTED



--- RACE (All group pairs) ---


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Amer-Indian-Eskimo,Asian-Pac-Islander,1,0,0.00,NO Disparate Treatment
1,Amer-Indian-Eskimo,Black,5,2,40.00,DETECTED
2,Amer-Indian-Eskimo,Other,1,0,0.00,NO Disparate Treatment
3,Amer-Indian-Eskimo,White,39,6,15.38,DETECTED
4,Asian-Pac-Islander,Black,2,0,0.00,NO Disparate Treatment
5,Asian-Pac-Islander,Other,2,1,50.00,DETECTED
6,Asian-Pac-Islander,White,70,5,7.14,DETECTED
7,Black,Other,2,0,0.00,NO Disparate Treatment
8,Black,White,340,22,6.47,DETECTED
9,Other,White,23,0,0.00,NO Disparate Treatment


In [ ]:
from itertools import combinations

non_sensitive_cols = ['Age', 'Workclass', 'Education', 'Martial_Status',
                      'Occupation', 'Relationship', 'Hours_per_week', 'Country']

sex_encoded_map  = {'Male': 1, 'Female': 0}
race_display_map = {0: 'Amer-Indian-Eskimo', 1: 'Asian-Pac-Islander',
                    2: 'Black', 3: 'Other', 4: 'White'}

test_df = X_test.copy().reset_index(drop=True)
test_df['Sex']  = sens_test.reset_index(drop=True)['Sex']
test_df['Race'] = sens_test.reset_index(drop=True)['Race']

def run_disparate_treatment(model, X_test_scaled, model_name):
    test_df['prediction'] = model.predict(X_test_scaled)
    sex_groups = test_df.groupby(non_sensitive_cols)

    print(f"\n{'='*60}")
    print(f"  {model_name} — Disparate Treatment")
    print(f"{'='*60}")

    # ── SEX ───────────────────────────────────────────────────
    print("\n--- SEX (Male vs Female) ---")
    total_pairs     = 0
    diff_pred_pairs = 0

    for _, group in sex_groups:
        males   = group[group['Sex'] == sex_encoded_map['Male']]
        females = group[group['Sex'] == sex_encoded_map['Female']]
        if len(males) > 0 and len(females) > 0:
            for _, m in males.iterrows():
                for _, f in females.iterrows():
                    total_pairs += 1
                    if m['prediction'] != f['prediction']:
                        diff_pred_pairs += 1

    pct     = (diff_pred_pairs / total_pairs * 100) if total_pairs > 0 else 0
    verdict = "DETECTED" if diff_pred_pairs > 0 else "NO Disparate Treatment"

    display(pd.DataFrame([{
        'Group 1'    : 'Male',
        'Group 2'    : 'Female',
        'Total Pairs': total_pairs,
        'Diff Preds' : diff_pred_pairs,
        '% Affected' : round(pct, 2),
        'Verdict'    : verdict
    }]))

    # ── RACE ──────────────────────────────────────────────────
    print("\n--- RACE (All group pairs) ---")
    race_groups_list = sorted(test_df['Race'].unique())
    race_pairs_list  = list(combinations(race_groups_list, 2))

    race_summary = []
    for r1, r2 in race_pairs_list:
        total_r = 0
        diff_r  = 0
        for _, group in sex_groups:
            g1 = group[group['Race'] == r1]
            g2 = group[group['Race'] == r2]
            if len(g1) > 0 and len(g2) > 0:
                for _, row1 in g1.iterrows():
                    for _, row2 in g2.iterrows():
                        total_r += 1
                        if row1['prediction'] != row2['prediction']:
                            diff_r += 1
        pct_r     = (diff_r / total_r * 100) if total_r > 0 else 0
        verdict_r = "DETECTED" if diff_r > 0 else "NO Disparate Treatment"
        race_summary.append({
            'Group 1'    : race_display_map[r1],
            'Group 2'    : race_display_map[r2],
            'Total Pairs': total_r,
            'Diff Preds' : diff_r,
            '% Affected' : round(pct_r, 2),
            'Verdict'    : verdict_r
        })

    display(pd.DataFrame(race_summary))

run_disparate_treatment(lr_with, X_test_with, "Logistic Regression")
run_disparate_treatment(rf_with, X_test_with, "Random Forest")
run_disparate_treatment(dt_with, X_test_with, "Decision Tree")


  Logistic Regression — Disparate Treatment

--- SEX (Male vs Female) ---


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Male,Female,799,16,2.0,DETECTED



--- RACE (All group pairs) ---


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Amer-Indian-Eskimo,Asian-Pac-Islander,1,0,0.00,NO Disparate Treatment
1,Amer-Indian-Eskimo,Black,5,0,0.00,NO Disparate Treatment
2,Amer-Indian-Eskimo,Other,1,0,0.00,NO Disparate Treatment
3,Amer-Indian-Eskimo,White,39,1,2.56,DETECTED
4,Asian-Pac-Islander,Black,2,0,0.00,NO Disparate Treatment
5,Asian-Pac-Islander,Other,2,0,0.00,NO Disparate Treatment
6,Asian-Pac-Islander,White,70,4,5.71,DETECTED
7,Black,Other,2,0,0.00,NO Disparate Treatment
8,Black,White,340,13,3.82,DETECTED
9,Other,White,23,0,0.00,NO Disparate Treatment



  Random Forest — Disparate Treatment

--- SEX (Male vs Female) ---


...

,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Amer-Indian-Eskimo,Asian-Pac-Islander,1,0,0.00,NO Disparate Treatment
1,Amer-Indian-Eskimo,Black,5,0,0.00,NO Disparate Treatment
2,Amer-Indian-Eskimo,Other,1,0,0.00,NO Disparate Treatment
3,Amer-Indian-Eskimo,White,39,1,2.56,DETECTED
4,Asian-Pac-Islander,Black,2,0,0.00,NO Disparate Treatment
5,Asian-Pac-Islander,Other,2,0,0.00,NO Disparate Treatment
6,Asian-Pac-Islander,White,70,4,5.71,DETECTED
7,Black,Other,2,0,0.00,NO Disparate Treatment
8,Black,White,340,25,7.35,DETECTED
9,Other,White,23,1,4.35,DETECTED



  Decision Tree — Disparate Treatment

--- SEX (Male vs Female) ---


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Male,Female,799,33,4.13,DETECTED



--- RACE (All group pairs) ---


,Group 1,Group 2,Total Pairs,Diff Preds,% Affected,Verdict
0,Amer-Indian-Eskimo,Asian-Pac-Islander,1,0,0.00,NO Disparate Treatment
1,Amer-Indian-Eskimo,Black,5,2,40.00,DETECTED
2,Amer-Indian-Eskimo,Other,1,0,0.00,NO Disparate Treatment
3,Amer-Indian-Eskimo,White,39,6,15.38,DETECTED
4,Asian-Pac-Islander,Black,2,0,0.00,NO Disparate Treatment
5,Asian-Pac-Islander,Other,2,1,50.00,DETECTED
6,Asian-Pac-Islander,White,70,5,7.14,DETECTED
7,Black,Other,2,0,0.00,NO Disparate Treatment
8,Black,White,340,22,6.47,DETECTED
9,Other,White,23,0,0.00,NO Disparate Treatment


# Disparate Impact — Ground Truth

In [ ]:
y_test_reset    = y_test.reset_index(drop=True)
sens_test_reset = sens_test.reset_index(drop=True)

sex_label_map  = {0: 'Female', 1: 'Male'}
race_label_map = {0: 'Amer-Indian-Eskimo', 1: 'Asian-Pac-Islander',
                  2: 'Black', 3: 'Other', 4: 'White'}

label_maps = {'Sex': sex_label_map, 'Race': race_label_map}

print("="*60)
print("  Disparate Impact — Ground Truth Analysis")
print("="*60)

for attr in ['Sex', 'Race']:
    print(f"\n--- {attr.upper()} ---")
    groups = sorted(sens_test_reset[attr].unique())
    rates  = {}

    for group in groups:
        label    = label_maps[attr][group]
        mask     = sens_test_reset[attr] == group
        total    = mask.sum()
        positive = y_test_reset[mask].sum()
        rate     = positive / total
        rates[label] = rate
        print(f"  P(>50K | {label:25s}) = {int(positive):4d} / {int(total):4d} = {rate:.4f}")

    max_group = max(rates, key=rates.get)
    min_group = min(rates, key=rates.get)

    print(f"\n  Highest rate : {max_group:25s} = {rates[max_group]:.4f}")
    print(f"  Lowest rate  : {min_group:25s} = {rates[min_group]:.4f}")
    print(f"  Difference   : {rates[max_group] - rates[min_group]:.4f}")
    print(f"  Verdict      : DISPARATE IMPACT DETECTED" if rates[max_group] - rates[min_group] > 0 else "  Verdict      : NO Disparate Impact")

    summary = pd.DataFrame([{
        'Group'       : label_maps[attr][g],
        'Total'       : int((sens_test_reset[attr] == g).sum()),
        'Actual >50K' : int(y_test_reset[sens_test_reset[attr] == g].sum()),
        'Rate'        : round(rates[label_maps[attr][g]], 4),
    } for g in groups])
    print()
    display(summary)

  Disparate Impact — Ground Truth Analysis

--- SEX ---
  P(>50K | Female                   ) =  235 / 1968 = 0.1194
  P(>50K | Male                     ) = 1267 / 4065 = 0.3117

  Highest rate : Male                      = 0.3117
  Lowest rate  : Female                    = 0.1194
  Difference   : 0.1923
  Verdict      : DISPARATE IMPACT DETECTED



,Group,Total,Actual >50K,Rate
0,Female,1968,235,0.1194
1,Male,4065,1267,0.3117



--- RACE ---
  P(>50K | Amer-Indian-Eskimo       ) =    6 /   64 = 0.0938
  P(>50K | Asian-Pac-Islander       ) =   44 /  186 = 0.2366
  P(>50K | Black                    ) =   80 /  559 = 0.1431
  P(>50K | Other                    ) =    3 /   38 = 0.0789
  P(>50K | White                    ) = 1369 / 5186 = 0.2640

  Highest rate : White                     = 0.2640
  Lowest rate  : Other                     = 0.0789
  Difference   : 0.1850
  Verdict      : DISPARATE IMPACT DETECTED



,Group,Total,Actual >50K,Rate
0,Amer-Indian-Eskimo,64,6,0.0938
1,Asian-Pac-Islander,186,44,0.2366
2,Black,559,80,0.1431
3,Other,38,3,0.0789
4,White,5186,1369,0.2640


# Disparate Impact
P(ŷ=1 | z=0) = P(ŷ=1 | z=1)
Rate of positive predictions must be equal across groups.

In [23]:
y_test_reset    = y_test.reset_index(drop=True)
sens_test_reset = sens_test.reset_index(drop=True)

sex_label_map  = {0: 'Female', 1: 'Male'}
race_label_map = {0: 'Amer-Indian-Eskimo', 1: 'Asian-Pac-Islander',
                  2: 'Black', 3: 'Other', 4: 'White'}

label_maps = {'Sex': sex_label_map, 'Race': race_label_map}

for model_name, y_pred in [("Logistic Regression", y_pred_lr),
                             ("Random Forest",       y_pred_rf),
                             ("Decision Tree",       y_pred_dt)]:
    print(f"\n{'='*55}\n  {model_name}\n{'='*55}")
    for attr in ['Sex', 'Race']:
        print(f"\n  {attr}:")
        rates = {}
        for group in sorted(sens_test_reset[attr].unique()):
            mask  = sens_test_reset[attr] == group
            rate  = y_pred[mask].mean()
            label = label_maps[attr][group]
            rates[label] = rate
            print(f"  P(ŷ=1 | {label:25s}) = {y_pred[mask].sum():4d} / {mask.sum():4d} = {rate:.4f}")
        diff = max(rates.values()) - min(rates.values())
        verdict = "DETECTED" if diff > 0 else "FAIR"
        print(f"  Difference = {max(rates.values()):.4f} - {min(rates.values()):.4f} = {diff:.4f} → {verdict}")


  Logistic Regression

  Sex:
  P(ŷ=1 | Female                   ) =   33 / 1968 = 0.0168
  P(ŷ=1 | Male                     ) =  859 / 4065 = 0.2113
  Difference = 0.2113 - 0.0168 = 0.1945 → DETECTED

  Race:
  P(ŷ=1 | Amer-Indian-Eskimo       ) =    2 /   64 = 0.0312
  P(ŷ=1 | Asian-Pac-Islander       ) =    6 /  186 = 0.0323
  P(ŷ=1 | Black                    ) =   22 /  559 = 0.0394
  P(ŷ=1 | Other                    ) =    0 /   38 = 0.0000
  P(ŷ=1 | White                    ) =  862 / 5186 = 0.1662
  Difference = 0.1662 - 0.0000 = 0.1662 → DETECTED

  Random Forest

  Sex:
  P(ŷ=1 | Female                   ) =  175 / 1968 = 0.0889
  P(ŷ=1 | Male                     ) = 1161 / 4065 = 0.2856
  Difference = 0.2856 - 0.0889 = 0.1967 → DETECTED

  Race:
  P(ŷ=1 | Amer-Indian-Eskimo       ) =    1 /   64 = 0.0156
  P(ŷ=1 | Asian-Pac-Islander       ) =   42 /  186 = 0.2258
  P(ŷ=1 | Black                    ) =   54 /  559 = 0.0966
  P(ŷ=1 | Other                    ) =    4 /   38 = 

# Disparate Mistreatment
P(ŷ ≠ y | z=0) = P(ŷ ≠ y | z=1)
Overall error rate must be equal across groups.

In [24]:
for model_name, y_pred in [("Logistic Regression", y_pred_lr),
                             ("Random Forest",       y_pred_rf),
                             ("Decision Tree",       y_pred_dt)]:
    print(f"\n{'='*55}\n  {model_name}\n{'='*55}")
    for attr in ['Sex', 'Race']:
        print(f"\n  {attr}:")
        errors = {}
        for group in sorted(sens_test_reset[attr].unique()):
            mask        = sens_test_reset[attr] == group
            label       = label_maps[attr][group]
            total       = mask.sum()
            wrong       = (y_pred[mask] != y_test_reset[mask]).sum()
            error_rate  = wrong / total
            errors[label] = error_rate
            print(f"  Error({label:25s}) = {wrong:4d} / {total:4d} = {error_rate:.4f}")
        diff    = max(errors.values()) - min(errors.values())
        verdict = "DETECTED" if diff > 0.05 else "FAIR"
        print(f"  Disparity = {max(errors.values()):.4f} - {min(errors.values()):.4f} = {diff:.4f} → {verdict}")


  Logistic Regression

  Sex:
  Error(Female                   ) =  226 / 1968 = 0.1148
  Error(Male                     ) = 1216 / 4065 = 0.2991
  Disparity = 0.2991 - 0.1148 = 0.1843 → DETECTED

  Race:
  Error(Amer-Indian-Eskimo       ) =    8 /   64 = 0.1250
  Error(Asian-Pac-Islander       ) =   44 /  186 = 0.2366
  Error(Black                    ) =   82 /  559 = 0.1467
  Error(Other                    ) =    3 /   38 = 0.0789
  Error(White                    ) = 1305 / 5186 = 0.2516
  Disparity = 0.2516 - 0.0789 = 0.1727 → DETECTED

  Random Forest

  Sex:
  Error(Female                   ) =  200 / 1968 = 0.1016
  Error(Male                     ) =  992 / 4065 = 0.2440
  Disparity = 0.2440 - 0.1016 = 0.1424 → DETECTED

  Race:
  Error(Amer-Indian-Eskimo       ) =    7 /   64 = 0.1094
  Error(Asian-Pac-Islander       ) =   28 /  186 = 0.1505
  Error(Black                    ) =   68 /  559 = 0.1216
  Error(Other                    ) =    3 /   38 = 0.0789
  Error(White         